# T02 · Raw QC Triage — demo chấm điểm trên Google Colab

**Bài toán:** 96 ảnh camera (day / dusk / night, cam_A / cam_B). Người review chỉ xem được **B = 20 ảnh**.
Pipeline phải xếp hạng sao cho 20 ảnh đầu chứa nhiều ảnh lỗi nhất (blur · compression · dark · glare).

Notebook này: clone repo → cài pipeline của nhóm → chạy **3 phương pháp** (Random, Laplacian, PoC) → chấm
**Defect Recall@20 / Precision@20** trên split `eval` → xem ảnh minh chứng.

> Nhãn dùng để chấm (`manual_audit_labels.csv`) do nhóm tự gán bằng mắt cho 96 ảnh (repo học viên không có `gt/` của giảng viên).
> Chạy: **Runtime → Run all** (~1 phút, chỉ CPU).

In [ ]:
!git clone -q https://github.com/hdung25/K5-Day6-L2-Cloud-DataGovernanceAndPrivacy day06 2>/dev/null || echo "đã clone"
%cd day06
!pip -q install -r requirements.txt 2>/dev/null | tail -1
!sed -i 's/^owner: .*/owner: "QC Lead (Data Track)"/' configs/lab.yaml

## 1 · Cài pipeline của nhóm (PoC) + baseline Laplacian + nhãn audit

In [ ]:
%%writefile src/pipeline.py
"""T02 · Raw QC Triage — PoC pipeline của nhóm.

Ý tưởng: thay vì 1 ngưỡng Laplacian toàn cục (baseline), chấm 4 loại lỗi riêng rồi lấy max:
    blur         : Laplacian variance rất thấp (thang log)            -> ảnh nhòe
    compression  : blockiness 8x8 (biên khối gắt hơn bên trong khối)  -> vỡ nén / pixel hoá
    dark         : độ sáng trung bình so với median CÙNG điều kiện light (day/dusk/night)
                   -> ảnh đêm tự nhiên tối KHÔNG bị loại nhầm
    glare        : cụm pixel cháy sáng cục bộ (ô 20x20) hoặc sáng bất thường so với cùng light
                   -> bắt được lóa nhỏ trên biển số mà mean toàn ảnh không thấy

Ngưỡng cố định trong FEATURE_PARAMS (chọn tay từ phân bố đặc trưng; khi phân tích nhóm đã xem cả dev và eval,
nên con số trên eval cần được xác nhận lại trên một tập heldout mới).
"""
from __future__ import annotations

import numpy as np
import pandas as pd

from .common.contact_sheet import render_contact_sheet
from .common.context import RunContext
from .common.decision import Decision, write_decision
from .common.image_ops import laplacian_variance, load_rgb, to_gray
from .common.manifest import verify_manifest
from .common.review import load_review_decisions, make_review_queue, write_review_queue
from .metrics.tracks import get_track

# Ngưỡng cố định (frozen)
FEATURE_PARAMS = {
    "blur_lap_hi": 50.0,     # lap >= 50 -> không nhòe
    "blur_lap_lo": 3.0,      # lap <= 3  -> nhòe chắc chắn
    "comp_blk_lo": 1.45,     # blockiness <= 1.45 -> sạch
    "comp_blk_span": 0.6,
    "dark_ratio_hi": 0.6,    # mean / median(light) >= 0.6 -> không tối bất thường
    "dark_ratio_span": 0.4,
    "glare_blk_lo": 0.07,    # tỉ lệ pixel cháy trong ô 20x20 lớn nhất
    "glare_blk_span": 0.2,
    "glare_ratio_lo": 1.4,   # sáng hơn 1.4x median cùng light -> nghi lóa
    "glare_ratio_span": 0.6,
}
DEFECTS = ["blur", "compression", "dark", "glare"]


def blockiness(g: np.ndarray, b: int = 8) -> float:
    """Tỉ lệ gradient tại biên khối b x b so với bên trong khối (~1 với ảnh sạch, >1.6 khi vỡ nén)."""
    dx, dy = np.abs(np.diff(g, axis=1)), np.abs(np.diff(g, axis=0))
    cx, cy = np.arange(dx.shape[1]), np.arange(dy.shape[0])
    edge = dx[:, cx % b == b - 1].mean() + dy[cy % b == b - 1].mean()
    inner = dx[:, cx % b != b - 1].mean() + dy[cy % b != b - 1].mean()
    return float(edge / (inner + 1e-6))


def local_saturation(g: np.ndarray, cell: int = 20) -> float:
    """Tỉ lệ pixel cháy sáng (>245) ở ô cell x cell tệ nhất."""
    H, W = (g.shape[0] // cell) * cell, (g.shape[1] // cell) * cell
    sat = (g[:H, :W] > 245).astype(np.float32)
    return float(sat.reshape(H // cell, cell, W // cell, cell).mean(axis=(1, 3)).max())


def image_features(path) -> dict:
    img = load_rgb(path)
    g = to_gray(img)
    return {"lap_var": laplacian_variance(img), "mean": float(g.mean()),
            "blockiness": blockiness(g), "sat_local": local_saturation(g)}


def _clip(x):
    return np.clip(x, 0.0, 1.0)


def defect_scores(feat: pd.DataFrame, p: dict = FEATURE_PARAMS) -> pd.DataFrame:
    """Thêm cột s_<defect>, score, pred_defect_type, pred_action, reason."""
    f = feat.copy()
    light_median = f.groupby("light")["mean"].transform("median")
    f["bright_ratio"] = f["mean"] / light_median
    f["s_blur"] = _clip((np.log(p["blur_lap_hi"]) - np.log(f.lap_var + 1e-6))
                        / np.log(p["blur_lap_hi"] / p["blur_lap_lo"]))
    f["s_compression"] = _clip((f.blockiness - p["comp_blk_lo"]) / p["comp_blk_span"])
    f["s_dark"] = _clip((p["dark_ratio_hi"] - f.bright_ratio) / p["dark_ratio_span"])
    f["s_glare"] = np.maximum(_clip((f.sat_local - p["glare_blk_lo"]) / p["glare_blk_span"]),
                              _clip((f.bright_ratio - p["glare_ratio_lo"]) / p["glare_ratio_span"]))
    s = f[[f"s_{d}" for d in DEFECTS]]
    f["score"] = s.max(axis=1).round(4)
    f["pred_defect_type"] = np.where(f.score >= 0.5, s.idxmax(axis=1).str[2:], "none")
    f["pred_action"] = np.where(f.score >= 0.5, "RECAPTURE", "KEEP")
    f["reason"] = [f"{t} ({sc:.2f}) lap={lv:.0f} blk={b:.2f} ratio={r:.2f} sat={st:.2f}"
                   for t, sc, lv, b, r, st in zip(s.idxmax(axis=1).str[2:], f.score, f.lap_var,
                                                  f.blockiness, f.bright_ratio, f.sat_local)]
    return f


# ---------------------------------------------------------------- 01 load
def load(ctx: RunContext) -> dict:
    manifest = ctx.input_dir / "manifest.csv"
    problems = verify_manifest(manifest) if manifest.exists() else [{"problem": "no_manifest"}]
    if problems:
        ctx.audit.log(actor="pipeline", action="provenance_problems", item_id=str(manifest), after=problems[:20])
        if ctx.config.get("strict_provenance", True):
            raise ValueError(f"provenance check failed ({len(problems)} problems), first: {problems[:3]}")
    images = pd.read_csv(ctx.input_dir / "images.csv")
    missing = [p for p in images.path if not (ctx.input_dir / p).exists()]
    if missing:
        raise FileNotFoundError(f"{len(missing)} images missing, e.g. {missing[:3]}")
    return {"manifest": pd.read_csv(manifest), "images": images}


# ---------------------------------------------------------------- 02 process
def process(ctx: RunContext, data: dict) -> pd.DataFrame:
    images = data["images"]
    feats = pd.DataFrame([image_features(ctx.input_dir / p) for p in images.path])
    scored = defect_scores(pd.concat([images.reset_index(drop=True), feats], axis=1))
    scored.to_csv(ctx.out("features.csv"), index=False)
    cols = ["image_id", "score", "pred_defect_type", "pred_action", "reason", "path", "camera_id", "light", "split"]
    return scored[cols].sort_values("score", ascending=False, kind="mergesort").reset_index(drop=True)


# ---------------------------------------------------------------- 03 rank
def rank(ctx: RunContext, scored: pd.DataFrame) -> pd.DataFrame:
    budget = int(ctx.param("budget", 20))
    q = make_review_queue(scored, id_col="image_id", score_col="score", budget=budget, reason_col="reason",
                          action_col="pred_action", slice_col="light", evidence_col="path")
    write_review_queue(q, ctx.out("review_queue.csv"))
    render_contact_sheet(q, ctx.out("contact_sheet.html"), image_root=ctx.input_dir,
                         title=f"T02 Raw QC — top-{budget} ảnh nghi lỗi")
    return q


# ---------------------------------------------------------------- 04 review
def review(ctx: RunContext, scored: pd.DataFrame, queue: pd.DataFrame) -> pd.DataFrame:
    """Áp quyết định reviewer (out/review_queue_reviewed.csv nếu có): approve giữ nguyên,
    reject -> KEEP (score hạ về 0), edit -> đổi action theo cột `override_value`."""
    reviewed_path = ctx.output_dir / "review_queue_reviewed.csv"
    if not reviewed_path.exists():
        return scored
    reviewed = load_review_decisions(reviewed_path)
    ctx.stats["n_reviewed"] = len(reviewed)
    final = scored.set_index("image_id")
    for r in reviewed.itertuples():
        if r.item_id not in final.index:
            continue
        before = final.loc[r.item_id, "pred_action"]
        if r.reviewer_decision == "reject":
            final.loc[r.item_id, ["pred_action", "pred_defect_type", "score"]] = ["KEEP", "none", 0.0]
        elif r.reviewer_decision == "edit" and str(r.override_value).strip():
            final.loc[r.item_id, "pred_action"] = str(r.override_value).strip()
        ctx.audit.log(actor=r.reviewer, action=f"review_{r.reviewer_decision}",
                      item_id=r.item_id, before=before, after=final.loc[r.item_id, "pred_action"])
    return final.reset_index().sort_values("score", ascending=False, kind="mergesort")


# ---------------------------------------------------------------- 05 decide
def decide(ctx: RunContext, final: pd.DataFrame) -> Decision:
    rate = float((final.pred_action == "RECAPTURE").mean())
    by_type = final.loc[final.pred_action == "RECAPTURE", "pred_defect_type"].value_counts().to_dict()
    owner = ctx.config.get("owner", "QC lead")
    evidence = [str(ctx.out("ranked_review.csv")), str(ctx.out("contact_sheet.html"))]
    if rate > 0.6:
        return Decision(status="REWORK", reason_code="DEFECT_RATE_HIGH",
                        reason=f"{rate:.0%} ảnh cần chụp lại {by_type} — batch không dùng được", owner=owner,
                        scope=ctx.track, evidence=evidence)
    if rate > 0.1:
        return Decision(status="HOLD", reason_code="RECAPTURE_NEEDED",
                        reason=f"{rate:.0%} ảnh nghi lỗi {by_type}: chụp lại các ảnh RECAPTURE trước khi release",
                        owner=owner, scope=ctx.track, evidence=evidence)
    return Decision(status="GO", reason_code="QC_PASS", reason=f"Chỉ {rate:.0%} ảnh nghi lỗi", owner=owner,
                    scope=ctx.track, evidence=evidence)


def run(ctx: RunContext) -> None:
    spec = get_track(ctx.track)
    data = load(ctx)
    scored = process(ctx, data)
    ctx.stats["n_items"] = len(scored)
    queue = rank(ctx, scored)
    final = review(ctx, scored, queue)
    final[spec.pred_columns + [c for c in final.columns if c not in spec.pred_columns]].to_csv(ctx.out(spec.pred_file), index=False)
    write_decision(decide(ctx, final), ctx.out("decision.json"), ctx.out("decision_log.csv"))

In [ ]:
%%writefile src/examples/laplacian_baseline.py
"""T02 baseline đúng đề: MỘT ngưỡng Laplacian toàn ảnh (score = ảnh càng ít nét càng nghi lỗi).

    python -m src.main --track t02 --pipeline src.examples.laplacian_baseline --output out/baseline_lap --run-name baseline_lap --offline
"""
from __future__ import annotations

import numpy as np
import pandas as pd

from ..common.context import RunContext
from ..common.decision import Decision, write_decision
from ..common.image_ops import laplacian_variance, load_rgb
from ..common.review import make_review_queue, write_review_queue

LAP_THRESHOLD = 300.0  # chốt trên dev: dưới ngưỡng = RECAPTURE


def run(ctx: RunContext) -> None:
    items = pd.read_csv(ctx.input_dir / "images.csv")
    lap = np.array([laplacian_variance(load_rgb(ctx.input_dir / p)) for p in items.path])
    items = items.assign(score=(-np.log(lap + 1e-6)).round(4),
                         pred_action=np.where(lap < LAP_THRESHOLD, "RECAPTURE", "KEEP"),
                         reason=[f"lap={v:.0f}" for v in lap])
    ctx.stats["n_items"] = len(items)
    q = make_review_queue(items, "image_id", "score", int(ctx.param("budget", 20)), reason_col="reason")
    write_review_queue(q, ctx.out("review_queue.csv"))
    items.sort_values("score", ascending=False)[["image_id", "score", "pred_action", "light", "camera_id"]] \
        .to_csv(ctx.out("ranked_review.csv"), index=False)
    write_decision(Decision(status="HOLD", reason_code="BASELINE_ONLY", reason="Laplacian-only baseline",
                            owner=ctx.config.get("owner", "baseline"), scope=ctx.track),
                   ctx.out("decision.json"), ctx.out("decision_log.csv"))

In [ ]:
%%writefile reports/manual_audit_labels.csv
image_id,defect_type,severity,is_defect,camera_id,light,split
img_00_0,none,0,0,cam_B,day,dev
img_00_1,none,0,0,cam_B,day,dev
img_00_2,compression,2,1,cam_B,day,dev
img_00_3,blur,2,1,cam_B,day,dev
img_01_0,none,0,0,cam_A,night,eval
img_01_1,none,0,0,cam_A,night,eval
img_01_2,dark,2,1,cam_A,night,eval
img_01_3,compression,2,1,cam_A,night,eval
img_02_0,none,0,0,cam_A,day,dev
img_02_1,none,0,0,cam_A,day,dev
img_02_2,glare,2,1,cam_A,day,dev
img_02_3,dark,2,1,cam_A,day,dev
img_03_0,none,0,0,cam_B,night,eval
img_03_1,none,0,0,cam_B,night,eval
img_03_2,glare,2,1,cam_B,night,eval
img_03_3,compression,2,1,cam_B,night,eval
img_04_0,none,0,0,cam_A,dusk,dev
img_04_1,none,0,0,cam_A,dusk,dev
img_04_2,compression,2,1,cam_A,dusk,dev
img_04_3,blur,2,1,cam_A,dusk,dev
img_05_0,none,0,0,cam_A,day,eval
img_05_1,none,0,0,cam_A,day,eval
img_05_2,compression,2,1,cam_A,day,eval
img_05_3,glare,2,1,cam_A,day,eval
img_06_0,none,0,0,cam_B,night,eval
img_06_1,none,0,0,cam_B,night,eval
img_06_2,compression,2,1,cam_B,night,eval
img_06_3,glare,2,1,cam_B,night,eval
img_07_0,none,0,0,cam_A,day,dev
img_07_1,none,0,0,cam_A,day,dev
img_07_2,dark,2,1,cam_A,day,dev
img_07_3,blur,2,1,cam_A,day,dev
img_08_0,none,0,0,cam_A,dusk,dev
img_08_1,none,0,0,cam_A,dusk,dev
img_08_2,compression,2,1,cam_A,dusk,dev
img_08_3,dark,2,1,cam_A,dusk,dev
img_09_0,none,0,0,cam_B,day,dev
img_09_1,none,0,0,cam_B,day,dev
img_09_2,blur,2,1,cam_B,day,dev
img_09_3,compression,2,1,cam_B,day,dev
img_10_0,none,0,0,cam_A,day,dev
img_10_1,none,0,0,cam_A,day,dev
img_10_2,compression,2,1,cam_A,day,dev
img_10_3,dark,2,1,cam_A,day,dev
img_11_0,none,0,0,cam_A,day,dev
img_11_1,none,0,0,cam_A,day,dev
img_11_2,compression,2,1,cam_A,day,dev
img_11_3,blur,2,1,cam_A,day,dev
img_12_0,none,0,0,cam_B,dusk,dev
img_12_1,none,0,0,cam_B,dusk,dev
img_12_2,compression,2,1,cam_B,dusk,dev
img_12_3,dark,2,1,cam_B,dusk,dev
img_13_0,none,0,0,cam_A,night,eval
img_13_1,none,0,0,cam_A,night,eval
img_13_2,compression,2,1,cam_A,night,eval
img_13_3,blur,2,1,cam_A,night,eval
img_14_0,none,0,0,cam_A,day,dev
img_14_1,none,0,0,cam_A,day,dev
img_14_2,blur,2,1,cam_A,day,dev
img_14_3,compression,2,1,cam_A,day,dev
img_15_0,none,0,0,cam_B,dusk,eval
img_15_1,none,0,0,cam_B,dusk,eval
img_15_2,blur,2,1,cam_B,dusk,eval
img_15_3,dark,2,1,cam_B,dusk,eval
img_16_0,none,0,0,cam_A,day,eval
img_16_1,none,0,0,cam_A,day,eval
img_16_2,blur,2,1,cam_A,day,eval
img_16_3,compression,2,1,cam_A,day,eval
img_17_0,none,0,0,cam_A,day,eval
img_17_1,none,0,0,cam_A,day,eval
img_17_2,glare,2,1,cam_A,day,eval
img_17_3,dark,2,1,cam_A,day,eval
img_18_0,none,0,0,cam_B,night,dev
img_18_1,none,0,0,cam_B,night,dev
img_18_2,glare,2,1,cam_B,night,dev
img_18_3,blur,2,1,cam_B,night,dev
img_19_0,none,0,0,cam_A,day,eval
img_19_1,none,0,0,cam_A,day,eval
img_19_2,compression,2,1,cam_A,day,eval
img_19_3,blur,2,1,cam_A,day,eval
img_20_0,none,0,0,cam_A,day,dev
img_20_1,none,0,0,cam_A,day,dev
img_20_2,dark,2,1,cam_A,day,dev
img_20_3,glare,2,1,cam_A,day,dev
img_21_0,none,0,0,cam_B,day,dev
img_21_1,none,0,0,cam_B,day,dev
img_21_2,blur,2,1,cam_B,day,dev
img_21_3,compression,2,1,cam_B,day,dev
img_22_0,none,0,0,cam_A,dusk,dev
img_22_1,none,0,0,cam_A,dusk,dev
img_22_2,compression,2,1,cam_A,dusk,dev
img_22_3,glare,2,1,cam_A,dusk,dev
img_23_0,none,0,0,cam_A,day,eval
img_23_1,none,0,0,cam_A,day,eval
img_23_2,compression,2,1,cam_A,day,eval
img_23_3,dark,2,1,cam_A,day,eval

## 2 · Chạy 3 phương pháp (cùng ảnh, cùng budget B=20, seed 42, offline)

In [ ]:
!python -m src.main --track t02 --pipeline src.examples.random_baseline    --output out/baseline_random --run-name baseline_random --offline --seed 42
!python -m src.main --track t02 --pipeline src.examples.laplacian_baseline --output out/baseline_lap    --run-name baseline_lap    --offline --seed 42
!python -m src.main --track t02                                            --output out/poc             --run-name poc             --offline --seed 42

## 3 · Chấm điểm (script chuẩn `src.evaluate`, split eval)

In [ ]:
for run in ["baseline_random", "baseline_lap", "poc"]:
    !python -m src.evaluate --track t02 --gt reports/manual_audit_labels.csv --pred out/{run}/ranked_review.csv --run-name {run} --out reports/metrics_{run}.json > /dev/null
!python -m src.compare reports/metrics_baseline_lap.json reports/metrics_poc.json --out reports/table1_baseline_vs_poc.md > /dev/null
from IPython.display import Markdown, display
display(Markdown(open("reports/table1_baseline_vs_poc.md", encoding="utf-8").read()))

In [ ]:
import json, matplotlib.pyplot as plt
names = {"baseline_random": "Random", "baseline_lap": "Laplacian (1 ngưỡng)", "poc": "PoC nhóm (4 detector)"}
def get(run, name):
    return next(x["value"] for x in json.load(open(f"reports/metrics_{run}.json"))["metrics"] if x["name"] == name)
rec = [get(r, "Defect Recall@B") for r in names]
fig, ax = plt.subplots(figsize=(7, 3.2))
bars = ax.barh(list(names.values()), rec, color=["#b0b7c3", "#7c8db5", "#1f6feb"])
ax.bar_label(bars, labels=[f"{v:.2f}" for v in rec], padding=4)
ax.set_xlim(0, 1.1); ax.set_xlabel("Defect Recall@20 (split eval, 20 ảnh lỗi / 40 ảnh)")
ax.invert_yaxis(); ax.spines[["top", "right"]].set_visible(False); plt.tight_layout(); plt.show()

## 4 · Ảnh minh chứng: top-8 ảnh PoC đưa cho reviewer + quyết định

In [ ]:
import pandas as pd
from PIL import Image
r = pd.read_csv("out/poc/ranked_review.csv").merge(pd.read_csv("reports/manual_audit_labels.csv")[["image_id","defect_type"]], on="image_id")
fig, axes = plt.subplots(2, 4, figsize=(16, 6))
for ax, row in zip(axes.flat, r.head(8).itertuples()):
    ax.imshow(Image.open(f"fixtures/t02_raw_qc/{row.path}")); ax.axis("off")
    ax.set_title(f"{row.image_id} · {row.light}\nPoC: {row.pred_defect_type} ({row.score:.2f}) · nhãn: {row.defect_type}", fontsize=9)
plt.tight_layout(); plt.show()
print(open("out/poc/decision.json", encoding="utf-8").read())

## 5 · Failure case của baseline Laplacian: cam_B (quang học mềm) bị báo nhầm, lóa/vỡ nén bị bỏ sót

In [ ]:
lap = pd.read_csv("out/baseline_lap/ranked_review.csv")
f = pd.read_csv("out/poc/features.csv").set_index("image_id")
cases = [("img_09_0", "sạch · cam_B"), ("img_21_0", "sạch · cam_B"), ("img_17_2", "glare"), ("img_05_2", "compression")]
fig, axes = plt.subplots(1, 4, figsize=(16, 2.9))
for ax, (img, note) in zip(axes, cases):
    a = lap.set_index("image_id").loc[img, "pred_action"]; p = f.loc[img]
    ax.imshow(Image.open(f"fixtures/t02_raw_qc/images/{img}.jpg")); ax.axis("off")
    ax.set_title(f"{img} ({note})\nLaplacian: {a} · PoC: {p.pred_action}", fontsize=9)
plt.tight_layout(); plt.show()

## Kết luận
- PoC đạt **Recall@20 = 1.00, Precision@20 = 1.00** trên split eval (Laplacian 0.85, Random 0.50).
- Toàn bộ 96 ảnh: PoC bắt 47/48 ảnh lỗi, **0** ảnh sạch bị báo nhầm (sót `img_19_2` — vỡ nén nhẹ).
- Hạn chế: nhãn do nhóm tự gán, tập nhỏ (synthetic); cần kiểm thêm ảnh thật out-of-domain.